# Personal Finance Anomaly Detection — Full Pipeline Demo

This notebook walks through every stage of the backend pipeline:

1. **Data Loading & PDF/CSV Parsing**
2. **Data Preprocessing & Cleaning**
3. **Auto-Categorization**
4. **Feature Engineering**
5. **Behavioural Baseline Computation**
6. **Hybrid Anomaly Detection (Statistical + Isolation Forest)**
7. **Explanation Generation**
8. **Model Persistence & Reloading**
9. **Visualization**

---
## 0. Setup & Imports

In [2]:
import sys, os
import warnings
warnings.filterwarnings('ignore')

# Ensure the app package is importable
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.ensemble import IsolationForest

# Our backend modules
from app.services.parser import parse_csv, parse_pdf
from app.services.categorizer import categorize, categorize_dataframe, CATEGORY_KEYWORDS
from app.services.feature_engineering import engineer_features, get_feature_matrix
from app.services.baseline import compute_baseline
from app.services.anomaly_engine import detect_anomalies
from app.services.explanation_engine import generate_explanations
from app.utils.helpers import clean_currency, extract_merchant, ensure_ml_models_dir

print("All imports successful ✓")

All imports successful ✓


---
## 1. Data Loading & Parsing

The parser accepts **CSV** or **PDF** bank statements.  
CSV expects columns: `date`, `description`, `amount` (flexible aliases supported).  
PDF uses pdfplumber table extraction with regex fallback.

Below we create a realistic sample dataset with timestamps to demonstrate all features.

In [ ]:
# Create a realistic sample CSV in-memory
csv_data = """date,description,amount
2025-01-02 10:15:00,Swiggy Food Delivery,450
2025-01-03 08:30:00,Uber Ride to Office,280
2025-01-04 14:20:00,Zomato Lunch Order,340
2025-01-05 19:45:00,Amazon Purchase - Headphones,3500
2025-01-06 20:00:00,Netflix Monthly Subscription,499
2025-01-07 12:10:00,Swiggy Dinner,680
2025-01-08 09:00:00,Electricity Bill Payment,2300
2025-01-09 11:30:00,Uber Ride Home,350
2025-01-10 17:45:00,Swiggy Lunch,220
2025-01-11 13:00:00,Flipkart Shopping - Shoes,4200
2025-01-12 10:00:00,Starbucks Coffee,550
2025-01-13 22:30:00,Zomato Late Night Order,890
2025-01-14 09:15:00,Ola Cab Ride,200
2025-01-15 11:00:00,Spotify Premium,129
2025-01-16 15:30:00,Uber Ride to Mall,310
2025-01-17 18:00:00,Rent Payment - Monthly,25000
2025-01-18 10:30:00,Swiggy Breakfast,190
2025-01-19 03:15:00,Unknown Online Purchase,8500
2025-01-20 14:00:00,Amazon Prime Purchase - TV,45000
2025-01-21 16:45:00,Zomato Party Order,4500
2025-01-22 10:00:00,Swiggy Regular Lunch,280
2025-01-23 02:30:00,ATM Cash Withdrawal,15000
2025-01-24 11:15:00,Uber Eats Dinner,650
2025-01-25 19:00:00,Movie Tickets PVR,1200
2025-01-26 12:00:00,Grocery Store BigBasket,2800
2025-01-27 09:30:00,Petrol HP Station,3200
2025-01-28 04:00:00,Suspicious Merchant ABC,50000
2025-01-29 13:00:00,Mobile Recharge Jio,999
2025-01-30 10:45:00,Swiggy Snacks,150
2025-01-31 20:30:00,Zomato Dinner,720
"""

# Parse using our CSV parser
df_raw = parse_csv(csv_data.encode('utf-8'))

print(f"Parsed {len(df_raw)} transactions")
print(f"Date range: {df_raw['date'].min()} → {df_raw['date'].max()}")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head(10)

NameError: name 'parse_csv' is not defined

### 1b. PDF Parsing (reference)

For PDF files, the parser works similarly:

```python
with open('bank_statement.pdf', 'rb') as f:
    df_pdf = parse_pdf(f.read())
```

It uses **pdfplumber** to extract tables, with a **regex fallback** for unstructured PDFs.  
The regex pattern matches: `date  description  amount` per line.

In [ ]:
# Show the PDF parser's regex pattern and table extraction logic
from app.services.parser import _PDF_ROW_RE

print("PDF regex pattern:")
print(f"  {_PDF_ROW_RE.pattern}\n")

# Test the regex on sample PDF-like text lines
sample_lines = [
    "01/15/2025  Swiggy Food Order  $450.00",
    "01/16/2025  Amazon Purchase  ₹12,500",
    "01/17/2025  Uber Ride  280.50",
]

print("Regex extraction test:")
for line in sample_lines:
    match = _PDF_ROW_RE.search(line)
    if match:
        print(f"  ✓ date={match.group(1)}, desc={match.group(2).strip()}, amt={match.group(3).strip()}")
    else:
        print(f"  ✗ No match: {line}")

---
## 2. Data Preprocessing & Cleaning

- Currency symbols stripped (`$`, `₹`, `,`)
- Dates parsed to datetime
- Missing values dropped
- Merchant names extracted from descriptions
- Sorted chronologically

In [ ]:
# Currency cleaning examples
test_values = ["$1,234.56", "₹ 500", "-$12.30", "45000", "  3,200.00  "]
print("Currency cleaning:")
for v in test_values:
    print(f"  '{v}' → {clean_currency(v)}")

print()

# Merchant extraction examples
test_descs = [
    "UPI/Swiggy Food Delivery/Payment",
    "NEFT Transfer to Amazon Pvt Ltd",
    "POS Debit Card Starbucks Coffee",
    "ATM Cash Withdrawal",
]
print("Merchant extraction:")
for d in test_descs:
    print(f"  '{d}' → '{extract_merchant(d)}'")

In [ ]:
# Apply merchant extraction to our data
df = df_raw.copy()
df['merchant'] = df['description'].apply(extract_merchant)

print("Extracted merchants:")
df[['description', 'merchant']].head(10)

---
## 3. Auto-Categorization

Keyword-based classification into: **Food, Transport, Shopping, Subscription, Housing, Entertainment, Bills, Transfer, Others**.

Easily extendable — just add keywords to the dictionary.

In [ ]:
# Show the keyword mapping
print("Category keywords:\n")
for cat, keywords in CATEGORY_KEYWORDS.items():
    print(f"  {cat:15s} → {', '.join(keywords[:6])}{'...' if len(keywords) > 6 else ''}")

In [ ]:
# Categorize all transactions
df = categorize_dataframe(df)

print("Category distribution:\n")
cat_counts = df['category'].value_counts()
for cat, count in cat_counts.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f"  {cat:15s} {count:3d}  ({pct:5.1f}%)  {bar}")

print(f"\nTotal: {len(df)} transactions")

In [ ]:
# Show categorization results
df[['description', 'category', 'amount']].head(15)

---
## 4. Feature Engineering

From 3 raw columns (`date`, `description`, `amount`), we derive **7 features**:

| Feature | Source | Description |
|---------|--------|-------------|
| `abs_amount` | amount | Absolute transaction value |
| `hour_of_day` | date | Hour (0-23) |
| `day_of_week` | date | 0=Mon, 6=Sun |
| `days_since_last_transaction` | date | Gap between consecutive transactions |
| `rolling_7_day_spend` | date + amount | 7-day trailing spend window |
| `merchant_frequency` | description | How many times this merchant appears |
| `category_frequency` | description | How many transactions in this category |

In [ ]:
# Run feature engineering
df_features = engineer_features(df)

feature_cols = [
    'abs_amount', 'hour_of_day', 'day_of_week',
    'days_since_last_transaction', 'rolling_7_day_spend',
    'merchant_frequency', 'category_frequency'
]

print(f"Features computed: {feature_cols}\n")
df_features[['description', 'amount'] + feature_cols].head(15)

In [ ]:
# Feature statistics
print("Feature Statistics:\n")
df_features[feature_cols].describe().round(2)

In [ ]:
# Visualise feature distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature Distributions', fontsize=14, fontweight='bold')

plot_features = ['abs_amount', 'hour_of_day', 'day_of_week',
                 'days_since_last_transaction', 'rolling_7_day_spend', 'merchant_frequency']

for ax, feat in zip(axes.flatten(), plot_features):
    ax.hist(df_features[feat], bins=15, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(feat, fontsize=10)
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

---
## 5. Behavioural Baseline

Per-user baseline captures normal spending behaviour:

- **Per category**: mean amount, std, frequency per week
- **Per merchant**: count, mean amount
- **Global**: weekly average spend, weekly std

In [ ]:
# Compute baseline
baseline = compute_baseline(df_features)

print("=" * 60)
print("USER BEHAVIOURAL BASELINE")
print("=" * 60)

print(f"\n📊 Weekly avg spend:  ₹{baseline['weekly_avg_spend']:,.2f}")
print(f"📊 Weekly std spend:  ₹{baseline['weekly_std_spend']:,.2f}")

print(f"\n--- Category Stats ({len(baseline['category_stats'])} categories) ---")
for cat, stats in sorted(baseline['category_stats'].items()):
    print(f"  {cat:15s}  mean=₹{stats['mean_amount']:>10,.2f}  "
          f"std=₹{stats['std_amount']:>9,.2f}  "
          f"freq/week={stats['frequency_per_week']:.1f}  "
          f"count={stats['count']}")

print(f"\n--- Merchant Stats ({len(baseline['merchant_stats'])} merchants) ---")
for merchant, stats in sorted(baseline['merchant_stats'].items(), key=lambda x: -x[1]['count'])[:10]:
    print(f"  {merchant:30s}  count={stats['count']}  mean=₹{stats['mean_amount']:,.2f}")

---
## 6. Hybrid Anomaly Detection

### Two-layer approach:

**Layer 1 — Statistical Scores** (4 sub-scores, each 0–1):
- Amount z-score vs category mean/std (weight: 35%)
- Weekly spending deviation vs baseline (weight: 20%)
- New merchant flag (weight: 20%)
- Time deviation from user's median hour (weight: 25%)

**Layer 2 — Isolation Forest** trained on:
- `abs_amount`, `hour_of_day`, `days_since_last_transaction`, `rolling_7_day_spend`

**Combined score:**
$$\text{risk\_score} = (0.6 \times \text{ML\_score} + 0.4 \times \text{statistical\_score}) \times 100$$

Normalised to **0–100**.

In [ ]:
# ── Layer 2: Isolation Forest — train and inspect ──

# Extract feature matrix (what the IF model sees)
feature_matrix = get_feature_matrix(df_features)
print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"Features used by IF: ['abs_amount', 'hour_of_day', 'days_since_last_transaction', 'rolling_7_day_spend']")
print(f"\nFirst 5 rows of feature matrix:")
print(pd.DataFrame(
    feature_matrix[:5],
    columns=['abs_amount', 'hour_of_day', 'days_since_last', 'rolling_7d_spend']
).to_string())

# Train a standalone IF to inspect
iso_forest = IsolationForest(
    n_estimators=100,
    contamination='auto',
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(feature_matrix)

# Raw scores (higher = more normal)
raw_scores = iso_forest.decision_function(feature_matrix)
predictions = iso_forest.predict(feature_matrix)  # 1=normal, -1=anomaly

print(f"\n--- Isolation Forest Results ---")
print(f"Model: IsolationForest(n_estimators=100, contamination='auto')")
print(f"Raw score range: [{raw_scores.min():.4f}, {raw_scores.max():.4f}]")
print(f"IF predictions: {(predictions == -1).sum()} anomalies, {(predictions == 1).sum()} normal")

In [ ]:
# ── Full hybrid detection ──

# Run the complete pipeline
df_result = detect_anomalies(
    df_features,
    baseline,
    user_id='notebook_demo',
    threshold=45  # lower threshold to see more flags for demo
)

print(f"Total transactions: {len(df_result)}")
print(f"Anomalies flagged:  {df_result['is_anomaly'].sum()}")
print(f"\nRisk score distribution:")
print(df_result['risk_score'].describe().round(2))

print(f"\n{'─'*80}")
print(f"{'Description':40s} {'Amount':>10s} {'Risk':>6s} {'ML':>6s} {'Stat':>6s} {'Flag':>5s}")
print(f"{'─'*80}")
for _, row in df_result.sort_values('risk_score', ascending=False).iterrows():
    flag = '🔴' if row['is_anomaly'] else '🟢'
    print(f"{row['description'][:40]:40s} ₹{row['abs_amount']:>9,.0f} "
          f"{row['risk_score']:>5.1f} {row['ml_score']:>5.3f} "
          f"{row['statistical_score']:>5.3f}  {flag}")

In [ ]:
# Visualise: Scatter plot of risk scores
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Risk score by amount
colors = ['red' if a else 'steelblue' for a in df_result['is_anomaly']]
axes[0].scatter(df_result['abs_amount'], df_result['risk_score'], c=colors, alpha=0.7, s=60)
axes[0].axhline(y=45, color='orange', linestyle='--', label='Threshold (45)')
axes[0].set_xlabel('Amount (₹)')
axes[0].set_ylabel('Risk Score')
axes[0].set_title('Risk Score vs Amount')
axes[0].legend()

# 2. ML score vs Statistical score
axes[1].scatter(df_result['ml_score'], df_result['statistical_score'], c=colors, alpha=0.7, s=60)
axes[1].set_xlabel('ML Score (Isolation Forest)')
axes[1].set_ylabel('Statistical Score')
axes[1].set_title('ML vs Statistical Score')

# 3. Risk score over time
axes[2].bar(range(len(df_result)), df_result['risk_score'], color=colors, alpha=0.8)
axes[2].axhline(y=45, color='orange', linestyle='--', label='Threshold')
axes[2].set_xlabel('Transaction Index (chronological)')
axes[2].set_ylabel('Risk Score')
axes[2].set_title('Risk Score Timeline')
axes[2].legend()

plt.tight_layout()
plt.show()

### 6b. Understanding the Statistical Sub-Scores

In [ ]:
# Break down the 4 statistical sub-scores for flagged transactions
flagged = df_result[df_result['is_anomaly'] == True].copy()

if len(flagged) > 0:
    print(f"{'Description':35s} {'Amt Z':>7s} {'WeekDev':>8s} {'NewMerch':>9s} {'TimeDev':>8s} {'Combined':>9s}")
    print('─' * 85)
    for _, row in flagged.iterrows():
        print(f"{row['description'][:35]:35s} "
              f"{row['stat_amount_zscore']:>6.3f} "
              f"{row['stat_weekly_dev']:>7.3f} "
              f"{row['stat_new_merchant']:>8.3f} "
              f"{row['stat_time_dev']:>7.3f} "
              f"{row['statistical_score']:>8.3f}")

    print(f"\nWeights: amount_zscore=0.35, weekly_dev=0.20, new_merchant=0.20, time_dev=0.25")
    print(f"Formula: statistical_score = 0.35 × amt_z + 0.20 × week_dev + 0.20 × new_merch + 0.25 × time_dev")
else:
    print("No anomalies flagged. Try lowering the threshold.")

---
## 7. Explanation Generation

For each flagged transaction, generate **human-readable explanations** explaining *why* it was flagged.

In [ ]:
# Generate explanations
explanations = generate_explanations(df_result, baseline)

print(f"Explanations generated for {len(explanations)} anomalous transactions:\n")

for result in explanations:
    # Find the matching row for context
    row = df_result[df_result.index == result.transaction_id - 1]
    desc = row['description'].values[0] if len(row) > 0 else 'N/A'
    amt = row['abs_amount'].values[0] if len(row) > 0 else 0

    print(f"🔴 Transaction #{result.transaction_id}")
    print(f"   Description: {desc}")
    print(f"   Amount: ₹{amt:,.0f}")
    print(f"   Risk Score: {result.risk_score}")
    print(f"   Explanations:")
    for exp in result.explanations:
        print(f"     → {exp}")
    print()

# Show the structured JSON output format
print("\n--- Structured JSON Output (API format) ---")
import json
for result in explanations[:3]:
    print(json.dumps(result.model_dump(), indent=2))

---
## 8. Model Persistence & Reloading

The Isolation Forest model is saved per-user as a `.pkl` file via **joblib**.  
This allows re-scoring new transactions without retraining.

In [ ]:
# The model was saved during detect_anomalies() — let's verify and reload it
from app.services.anomaly_engine import load_model
from app.utils.helpers import ML_MODELS_DIR

model_path = ML_MODELS_DIR / 'notebook_demo_model.pkl'
print(f"Model path: {model_path}")
print(f"Model exists: {model_path.exists()}")

if model_path.exists():
    file_size = model_path.stat().st_size
    print(f"Model size: {file_size / 1024:.1f} KB")

# Reload the model
loaded_model = load_model('notebook_demo')
print(f"\nLoaded model type: {type(loaded_model).__name__}")
print(f"Model parameters:")
for k, v in loaded_model.get_params().items():
    print(f"  {k}: {v}")

In [ ]:
# Score NEW transactions with the saved model (no retraining needed)
new_transactions = np.array([
    [500, 14, 1, 3000],     # Normal: ₹500 at 2pm, 1 day gap, ₹3k weekly
    [50000, 3, 0, 80000],   # Suspicious: ₹50k at 3am, same day, ₹80k weekly
    [200, 10, 2, 2000],     # Normal: ₹200 at 10am, 2 day gap, ₹2k weekly
    [35000, 2, 0, 95000],   # Suspicious: ₹35k at 2am, same day, ₹95k weekly
])

# Use loaded model to score
scores = loaded_model.decision_function(new_transactions)
predictions = loaded_model.predict(new_transactions)

print("Scoring new transactions with saved model:\n")
print(f"{'Amount':>10s} {'Hour':>5s} {'DayGap':>7s} {'WkSpend':>10s} {'Score':>8s} {'Verdict':>10s}")
print('─' * 55)
for i, (txn, score, pred) in enumerate(zip(new_transactions, scores, predictions)):
    verdict = '🔴 Anomaly' if pred == -1 else '🟢 Normal'
    print(f"₹{txn[0]:>9,.0f} {txn[1]:>5.0f} {txn[2]:>7.0f} ₹{txn[3]:>9,.0f} {score:>7.4f}  {verdict}")

---
## 9. Visualization — Full Analysis Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Personal Finance Anomaly Detection — Analysis Dashboard',
             fontsize=16, fontweight='bold', y=1.02)

colors_full = ['red' if a else 'steelblue' for a in df_result['is_anomaly']]

# 1. Spending by category (pie chart)
cat_spend = df_result.groupby('category')['abs_amount'].sum().sort_values(ascending=False)
axes[0, 0].pie(cat_spend, labels=cat_spend.index, autopct='%1.1f%%',
               startangle=90, colors=plt.cm.Set3.colors)
axes[0, 0].set_title('Spending by Category')

# 2. Transaction amounts over time
axes[0, 1].bar(range(len(df_result)), df_result['abs_amount'], color=colors_full, alpha=0.8)
axes[0, 1].set_xlabel('Transaction #')
axes[0, 1].set_ylabel('Amount (₹)')
axes[0, 1].set_title('Transaction Amounts (🔴 = Anomaly)')

# 3. Rolling 7-day spend
axes[1, 0].plot(df_result['rolling_7_day_spend'], color='steelblue', linewidth=2)
axes[1, 0].axhline(y=baseline['weekly_avg_spend'], color='green', linestyle='--',
                    label=f"Avg weekly: ₹{baseline['weekly_avg_spend']:,.0f}")
axes[1, 0].fill_between(
    range(len(df_result)),
    baseline['weekly_avg_spend'] - baseline['weekly_std_spend'],
    baseline['weekly_avg_spend'] + baseline['weekly_std_spend'],
    alpha=0.2, color='green', label='±1 std'
)
axes[1, 0].set_xlabel('Transaction #')
axes[1, 0].set_ylabel('₹')
axes[1, 0].set_title('Rolling 7-Day Spend vs Baseline')
axes[1, 0].legend()

# 4. Hour distribution (normal vs anomaly)
normal_hours = df_result[~df_result['is_anomaly']]['hour_of_day']
anomaly_hours = df_result[df_result['is_anomaly']]['hour_of_day']
axes[1, 1].hist(normal_hours, bins=24, range=(0, 24), alpha=0.7,
                color='steelblue', label='Normal', edgecolor='white')
axes[1, 1].hist(anomaly_hours, bins=24, range=(0, 24), alpha=0.7,
                color='red', label='Anomaly', edgecolor='white')
axes[1, 1].set_xlabel('Hour of Day')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Transaction Hour Distribution')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

---
## Summary

| Stage | Input | Output |
|-------|-------|--------|
| **Parsing** | CSV/PDF file bytes | DataFrame (date, description, amount) |
| **Preprocessing** | Raw DataFrame | Cleaned + merchant extracted |
| **Categorization** | description text | category label |
| **Feature Engineering** | 3 columns | 7 derived features |
| **Baseline** | Transaction history | Per-category/merchant/weekly stats |
| **Statistical Scoring** | Features + Baseline | 4 sub-scores → combined (0–1) |
| **ML Scoring** | Feature matrix | Isolation Forest score (0–1) |
| **Hybrid Score** | ML + Statistical | `0.6×ML + 0.4×Stat` → 0–100 |
| **Explanations** | Flagged transactions | Human-readable reasons |
| **Persistence** | Trained IF model | `.pkl` file per user |